# 02 — Hierarchical Bayesian MMM

Goal: fit a hierarchical MMM with partial pooling across 26 divisions.

**Input:** `data/processed/clean.parquet`, `data/processed/eda_stats.json`  
**Output:** `data/processed/trace.nc`, `data/processed/contributions.parquet`, `data/processed/model_summary.json`

> Run `01_eda.ipynb` first — channel/control lists are read from `eda_stats.json`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
az.style.use('arviz-darkgrid')

from src.mmm import BayesianMMM

PROC       = Path('../data/processed')
DATE_COL   = 'date'
DIV_COL    = 'Division'
TARGET_COL = 'Sales'

# Read channel and control lists from the artifact produced by 01_eda.ipynb
# so this notebook stays in sync when new features are added there.
with open(PROC / 'eda_stats.json') as f:
    eda = json.load(f)

CHANNEL_COLS = eda['channel_cols']
CONTROL_COLS = eda['control_cols']  # Organic_Views + calendar + holidays + macro

print('Channels :', CHANNEL_COLS)
print('Controls :', CONTROL_COLS)
print('Setup complete.')

## 1. Load clean data

In [ ]:
df = pd.read_parquet(PROC / 'clean.parquet')
print(df.shape)
print('Divisions:', df[DIV_COL].nunique())
print('Columns:',   list(df.columns))
df.head()

## 2. Sanity checks before modelling

In [ ]:
assert df[CHANNEL_COLS + CONTROL_COLS + [TARGET_COL]].isnull().sum().sum() == 0, \
    'Unexpected nulls — fix in 01_eda first'
assert df[TARGET_COL].min() > 0, 'Sales must be positive for log-normal likelihood'
print('All checks passed.')

## 3. Fit model

First run takes ~10–30 minutes. Once `trace.nc` exists, skip to the load cell at the bottom.

In [ ]:
mmm = BayesianMMM(
    channel_cols=CHANNEL_COLS,
    control_cols=CONTROL_COLS,
    date_col=DATE_COL,
    target_col=TARGET_COL,
)

idata = mmm.fit(
    df,
    draws=1000,
    tune=1000,
    target_accept=0.9,
    random_seed=42,
)
print('Sampling complete.')

## 4. Diagnostics — R-hat and ESS

In [ ]:
summary = az.summary(idata)
bad = summary[summary['r_hat'] > 1.05]
print(f'{len(bad)} parameters with R-hat > 1.05:')
print(bad[['mean', 'sd', 'r_hat', 'ess_bulk']].to_string())

In [ ]:
alpha_params = [p for p in idata.posterior.data_vars if 'alpha' in p]
if alpha_params:
    az.plot_trace(idata, var_names=alpha_params[:4])
    plt.tight_layout()

In [ ]:
az.plot_energy(idata)
plt.tight_layout()

## 5. Posterior predictive check

In [ ]:
mmm.mmm.sample_posterior_predictive(idata)
az.plot_ppc(idata, observed_rug=True)
plt.tight_layout()

## 6. Actual vs Fitted

In [ ]:
y_hat = idata.posterior_predictive['y'].mean(dim=['chain', 'draw']).values

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['date'].values, df[TARGET_COL].values, label='Actual', alpha=0.8)
ax.plot(df['date'].values, y_hat, label='Fitted (posterior mean)', linestyle='--', alpha=0.8)
ax.set_title('Actual vs Fitted — Sales')
ax.set_xlabel('Date')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
fig.savefig(PROC / 'fig_actual_vs_fitted.png', bbox_inches='tight')

## 7. Channel contributions

In [ ]:
contrib = mmm.channel_contribution_breakdown()
contrib.to_parquet(PROC / 'contributions.parquet', index=False)
print(contrib)

## 8. Save trace & model summary

In [ ]:
idata.to_netcdf(str(PROC / 'trace.nc'))
print('Trace saved → data/processed/trace.nc')

model_summary = {
    'channel_cols': CHANNEL_COLS,
    'control_cols': CONTROL_COLS,
    'target_col': TARGET_COL,
    'n_divergences': int(idata.sample_stats['diverging'].sum()),
    'max_rhat': float(summary['r_hat'].max()),
    'min_ess_bulk': float(summary['ess_bulk'].min()),
}
with open(PROC / 'model_summary.json', 'w') as f:
    json.dump(model_summary, f, indent=2)
print(json.dumps(model_summary, indent=2))

## (Optional) Load existing trace — skip fitting above

In [ ]:
# Uncomment to load a previously saved trace instead of re-sampling
# idata = az.from_netcdf(str(PROC / 'trace.nc'))
# mmm.idata = idata
# print('Trace loaded from disk.')